# 05. К3: К2 + FDL по картам границ

Приём RS-AdaDiff — перцептивная метрика между картами Собеля выхода и эталона; вместо DISTS здесь FDL. Запускать, если после К2 ошибка на контурах остаётся высокой.

## Инструкция

1. **Настройки ноутбука (панель справа).** Accelerator — `GPU T4 x2`, Internet — `On`, Persistence — `Files only`.
2. **Подключите данные (Add Input):**
   - обучающие HQ и валидационные HQ (LQ создаются в ноутбуке ячейкой `make_lq`);
   - датасет с Stable Diffusion 2.1-base (папки `unet`, `vae`, `tokenizer`, `text_encoder`) и релизный чекпоинт `pisa_sr.pkl`;
   - код берётся из GitHub (`git clone` в первой ячейке), поэтому Internet — `On`.
3. **Ячейки подготовки.** Первые ячейки клонируют код, собирают папку SD 2.1-base (в датасете нет `scheduler`, конфиг
   создаётся автоматически) и создают LQ. В ячейке «Пути» впишите свои пути; ячейка «Проверка» сразу скажет, чего не хватает.
4. **Запускайте ячейки по порядку.** Обучение печатает строку каждые 20 шагов и валидацию каждые `EVAL_EVERY` шагов (по умолчанию 500).
5. **Время.** Сессия Kaggle ограничена; если этап не успевает, уменьшите `STEPS` или запускайте этапы по отдельности
   (см. раздел «Если сессия оборвалась» в конце).
6. **Сохраните результат.** Последняя ячейка упаковывает логи и чекпоинты в архив. Нажмите `Save Version` →
   `Save & Run All` либо скачайте архив из `/kaggle/working`. Для сравнения экспериментов
   (ноутбук `08_compare_all`) подключите сохранённые результаты как входные данные.

**Что смотреть в результатах**

- `curve.png` и `curve_*.csv` — LPIPS, VoL/GT и ошибка в высокочастотных полосах по шагам;
- `hf_err` — главная метрика: 1 = детали не восстановлены, больше 1 = выдуманы, меньше — лучше;
- `eff_gsd_cm` — эффективное разрешение в сантиметрах (меньше — лучше);
- `train_log_*.csv` — значение, вес и норма градиента каждого члена потерь. Если норма градиента частотного
  члена меньше 5 % от главного (L2 в pix-этапе, LPIPS в sem-этапе), член почти не влияет — увеличьте вес.

In [ ]:
# Код из GitHub. Сначала уходим из папки репозитория, иначе rm -rf удалит текущий каталог ядра
%cd /kaggle/working
!rm -rf /kaggle/working/PiSA-SR
!git clone https://github.com/veronika14may/PiSA-SR
%cd /kaggle/working/PiSA-SR

In [ ]:
!mkdir -p /kaggle/working/sd_v2_1_full
!SRC=/kaggle/input/datasets/vende14/sd-v2-1/sd_v2_1; \
for d in unet vae tokenizer text_encoder; do ln -sf $SRC/$d /kaggle/working/sd_v2_1_full/$d; done; \
ls -la /kaggle/working/sd_v2_1_full

# В датасете sd_v2_1 нет папки scheduler — создаём конфиг SD 2.1-base вручную
import os, json
sched = "/kaggle/working/sd_v2_1_full/scheduler"
if os.path.islink(sched):      # остался битый симлинк с прошлого запуска
    os.remove(sched)
os.makedirs(sched, exist_ok=True)
json.dump({
    "_class_name": "PNDMScheduler", "beta_end": 0.012, "beta_schedule": "scaled_linear",
    "beta_start": 0.00085, "clip_sample": False, "num_train_timesteps": 1000,
    "prediction_type": "epsilon", "set_alpha_to_one": False, "skip_prk_steps": True,
    "steps_offset": 1, "trained_betas": None,
}, open(sched + "/scheduler_config.json", "w"), indent=2)

In [ ]:
NOTEBOOK_TAG = "05_K3"

# ==== Пути: отредактируйте под свои датасеты ====
CODE_SRC  = "/kaggle/working/PiSA-SR"                       # клон репозитория (внутри должна быть freq_exp)
SD_PATH   = "/kaggle/working/sd_v2_1_full"                  # Stable Diffusion 2.1-base
INIT_CKPT = "/kaggle/input/datasets/vende14/pisa-sr/pisa_sr.pkl"   # релизный чекпоинт PiSA-SR
RAM_PATH  = ""      # файл ram_swin_large_14m.pth; пусто = без RAM (фиксированный промпт)
TRAIN_HQ  = "/kaggle/input/datasets/vende14/data-from-scratch-netherlands/train/hq"
VAL_HQ    = "/kaggle/input/datasets/vende14/hq-lq-netherlands-val/hq_images"
TRAIN_LQ  = "/kaggle/working/lq_train"      # создаются ячейкой make_lq из HQ
VAL_LQ    = "/kaggle/working/lq_val"

# ==== Параметры обучения ====
OUT_ROOT = "/kaggle/working/experiments/freq"   # сюда пишутся логи и чекпоинты
STEPS    = 4000    # шагов на этап (как в ваших прежних запусках)
EVAL_EVERY = 500   # валидация каждые столько шагов; чаще — точнее кривая, но дольше обучение
ACCUM    = 1       # накопление градиента: 4 сглаживает кривые, но за те же шаги даёт вчетверо меньше обновлений
EMA      = 0       # 0.999 включает EMA весов LoRA (сохраняется отдельный *_ema.pkl)
FDL_CROP = 256     # окно для FDL: 256 экономит память; 0 = весь кроп 512
AUTO     = 0.3     # калибровка весов частотных членов по норме градиента; 0 = взять веса как есть

In [ ]:
# LQ из HQ без сдвига (повторный запуск пропускает готовые файлы)
!python /kaggle/working/PiSA-SR/freq_exp/make_lq.py --hq_dir {TRAIN_HQ} --out_dir {TRAIN_LQ}
!python /kaggle/working/PiSA-SR/freq_exp/make_lq.py --hq_dir {VAL_HQ} --out_dir {VAL_LQ} --seed 1

In [ ]:
import os, subprocess, sys

# --- Проверка ---
!nvidia-smi --query-gpu=name,memory.total --format=csv
missing = [p for p in [CODE_SRC, SD_PATH, INIT_CKPT, TRAIN_LQ, TRAIN_HQ, VAL_LQ, VAL_HQ] if not os.path.exists(p)]
assert not missing, f"Не найдены пути: {missing}"
assert os.path.isdir(os.path.join(CODE_SRC, "freq_exp")), "В клоне нет папки freq_exp"
if RAM_PATH and not os.path.exists(RAM_PATH):
    raise FileNotFoundError(RAM_PATH)
os.chdir(CODE_SRC)

# --- Пакеты: две функции потерь из статей + метрики ---
!pip install -q focal-frequency-loss fdl-pytorch lpips pyiqa
# peft требует torchao>=0.16, а на Kaggle стоит 0.10 — он не нужен
!pip uninstall -y -q torchao
# Если окружение чистое, раскомментируйте установку зависимостей PiSA-SR:
# !pip install -q -r requirements.txt

os.environ.update(SD=SD_PATH, INIT=INIT_CKPT, RAM=RAM_PATH, TRAIN_LQ=TRAIN_LQ, TRAIN_HQ=TRAIN_HQ,
                  VAL_LQ=VAL_LQ, VAL_HQ=VAL_HQ, OUT=OUT_ROOT, STEPS=str(STEPS), ACCUM=str(ACCUM),
                  EMA=str(EMA), FDL_CROP=str(FDL_CROP), AUTO=str(AUTO), EVAL_EVERY=str(EVAL_EVERY))
print("Рабочая папка:", os.getcwd())
!bash freq_exp/experiments.sh list

In [ ]:
import glob, pandas as pd
from IPython.display import Image, display

def show(exp_prefix):
    """Кривые, последние точки валидации и нормы градиентов для эксперимента."""
    dirs = sorted(glob.glob(os.path.join(OUT_ROOT, exp_prefix + "*")))
    assert dirs, f"нет результатов для {exp_prefix}"
    d = dirs[0]; print("==", d)
    if os.path.exists(os.path.join(d, "curve.png")):
        display(Image(os.path.join(d, "curve.png"), width=700))
    cols = ["global_step", "stage", "lpips_val", "vol_ratio", "hf_frc", "hf_err", "hf_energy", "eff_gsd_cm", "mem0_gb", "mem1_gb"]
    for f in sorted(glob.glob(os.path.join(d, "curve_*.csv"))):
        c = pd.read_csv(f); display(c[cols].tail(5))
    for f in sorted(glob.glob(os.path.join(d, "train_log_*.csv"))):
        t = pd.read_csv(f).dropna(axis=1, how="all")
        g = t[[c for c in t.columns if c.startswith(("global_step", "stage", "w_", "gnorm_"))]]
        print("Веса и нормы градиентов членов потерь (последние записи):"); display(g.tail(4))

In [ ]:
# Инференс итогового чекпоинта на валидации и оценка по частотным полосам.
N_EVAL = 100          # сколько снимков (0 = все 496; дольше)
def infer(exp_prefix, ckpt_name="pisasr_final.pkl"):
    d = sorted(glob.glob(os.path.join(OUT_ROOT, exp_prefix + "*")))[0]
    out = os.path.join("/kaggle/working/sr", os.path.basename(d))
    !python freq_exp/infer_eval.py --pretrained_model_path {SD_PATH} --ckpt {d}/checkpoints/{ckpt_name} \
        --lq_dir {VAL_LQ} --hq_dir {VAL_HQ} --out_dir {out} --n {N_EVAL}
    return os.path.join(out, "lsem_1.0")

## E10. FDL по картам Собеля

Отличие от исходного обучения PiSA-SR: `К2 + --sem_fdl_edge 0.0005`

Обучение разбито на две ячейки: сначала pix-этап, затем sem-этап. Pix-этап сохраняет `checkpoints/pisasr_pix_final.pkl`, с него стартует sem-этап.

In [ ]:
# pix-этап
!STAGE=pix bash freq_exp/experiments.sh E10

In [ ]:
# sem-этап (стартует с pisasr_pix_final.pkl)
!STAGE=sem bash freq_exp/experiments.sh E10

In [ ]:
show("E10")

In [ ]:
sr_E10 = infer("E10")

In [ ]:
# Архив с логами и итоговыми чекпоинтами (промежуточные чекпоинты не включаются).
import zipfile
zpath = "/kaggle/working/results_" + NOTEBOOK_TAG + ".zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT_ROOT):
        for fn in files:
            if fn.endswith((".csv", ".png", ".json")) or "final" in fn:
                p = os.path.join(root, fn); z.write(p, os.path.relpath(p, os.path.dirname(OUT_ROOT)))
    if os.path.isdir("/kaggle/working/sr"):
        for root, _, files in os.walk("/kaggle/working/sr"):
            for fn in files:
                if fn.endswith(".csv"):
                    p = os.path.join(root, fn); z.write(p, os.path.relpath(p, "/kaggle/working"))
print(zpath, round(os.path.getsize(zpath) / 2**20, 1), "МБ")

## Если сессия оборвалась

- **Оба этапа не помещаются в одну сессию.** В экспериментах с двумя этапами обучение уже разбито на две ячейки:
  `STAGE=pix` и `STAGE=sem`. Если сессия оборвалась после pix-этапа, не запускайте pix заново: сохраните версию
  ноутбука (`Save Version`), подключите её выход и положите `checkpoints/pisasr_pix_final.pkl` в папку
  `<OUT_ROOT>/<эксперимент>/checkpoints/`, затем запустите только ячейку `STAGE=sem`.
- **Не хватает памяти (CUDA out of memory).** Проверьте, что выбраны две видеокарты; оставьте `FDL_CROP = 256`;
  в крайнем случае добавьте к команде `--fdl_num_proj 64`.
- **Потеря стала NaN.** Чаще всего это Fourier Space Loss со слишком большим весом: оставьте `AUTO = 0.3`.